# 11 · Structured Streaming

So far our medallion pipeline ran in **batch**. **Structured Streaming** runs the
*same* DataFrame logic **continuously**, processing new data within seconds as it
arrives — for live dashboards, fraud detection, and real-time medallion pipelines.

The beautiful part: a streaming query is almost identical to a batch one. You
swap `spark.read` → `spark.readStream` and `df.write` → `df.writeStream`, and add
a **checkpoint** (for fault tolerance / exactly-once) and a **trigger**.

In [ ]:
try:
    spark
except NameError:
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.getOrCreate()
from pyspark.sql import functions as F
spark.sql("USE SCHEMA brewbox")
CATALOG = spark.sql("SELECT current_catalog()").first()[0]
LANDING = f"/Volumes/{CATALOG}/brewbox/landing"
print("landing:", LANDING)

## 1 · The streaming model — micro-batches

Structured Streaming treats a stream as an **unbounded table** that grows over
time. Spark processes it in small **micro-batches**, and the engine tracks
progress in a **checkpoint** so it can recover exactly-once after a failure.

Core pieces:
- **Source** — files (Auto Loader), Kafka, a rate generator, or a Delta table.
- **Transformations** — the same DataFrame ops as batch.
- **Sink** — Delta table, Kafka, etc., via `writeStream`.
- **Trigger** — how often to run a micro-batch (`processingTime`, `availableNow`,
  or default/continuous).
- **Output mode** — `append` (new rows), `update` (changed aggregates), `complete`
  (whole result each time).

## 2 · Stream files into Bronze with Auto Loader

Auto Loader (notebook 7) *is* a streaming source. Here it continuously ingests any
new `events` files into a **Bronze** table. `trigger(availableNow=True)` processes
everything available now and stops — the same code with a time trigger would run
forever, picking up files as they land.

In [ ]:
events_schema = f"{LANDING}/_schemas/events_stream"
events_ckpt   = f"{LANDING}/_checkpoints/events_bronze_stream"

q = (spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", events_schema)
    .load(f"{LANDING}/events")
    .withColumn("_ingested_at", F.current_timestamp())
    .writeStream.format("delta")
    .option("checkpointLocation", events_ckpt)
    .trigger(availableNow=True)
    .toTable("brewbox.events_bronze_stream"))
q.awaitTermination()
print("streamed events into Bronze:", spark.table("brewbox.events_bronze_stream").count())

## 3 · Streaming aggregation with a watermark

Aggregating a stream needs a **watermark** — it tells Spark how long to wait for
late data before finalizing a time window (and lets it drop old state so memory
stays bounded). Here we count events per type per 1-hour window.

In [ ]:
agg = (spark.readStream.table("brewbox.events_bronze_stream")
    .withColumn("event_time", F.to_timestamp("ts"))
    .withWatermark("event_time", "2 hours")
    .groupBy(F.window("event_time", "1 hour"), "event_type")
    .count())

qa = (agg.writeStream.format("delta")
    .option("checkpointLocation", f"{LANDING}/_checkpoints/events_hourly")
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable("brewbox.events_hourly"))
qa.awaitTermination()
spark.table("brewbox.events_hourly").orderBy("window").show(5, truncate=False)

## 4 · Upserts in a stream with `foreachBatch`

To `MERGE` streaming data into a target (e.g., maintain a Silver table), use
**`foreachBatch`** — it hands each micro-batch to a normal batch function where
you can run any Delta operation, including `MERGE`.

```python
def upsert_to_silver(micro_batch_df, batch_id):
    (DeltaTable.forName(spark, "brewbox.events_silver").alias("t")
        .merge(micro_batch_df.alias("s"), "t.event_id = s.event_id")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())

(stream_df.writeStream
    .foreachBatch(upsert_to_silver)
    .option("checkpointLocation", ckpt)
    .trigger(availableNow=True)
    .start())
```

This is how you build an **incremental, always-fresh Silver** from a stream.

## 5 · Streaming medallion & triggers

The whole medallion can be streaming: **Auto Loader → Bronze** (stream) →
**Bronze → Silver** (stream with `foreachBatch` MERGE) → **Silver → Gold**
(streaming aggregation). Trigger choices:

- `trigger(availableNow=True)` — process all available data once, then stop
  (great for scheduled "streaming batch" jobs; cost-efficient).
- `trigger(processingTime="1 minute")` — a micro-batch every minute (near-real-time).
- default — continuous micro-batches as fast as possible.

> Tip: on Databricks, **Lakeflow Declarative Pipelines (DLT)** — next notebook —
> can manage all this streaming plumbing for you declaratively.

## 6 · Exercises

**Exercise 1 —** Query `brewbox.events_hourly`: which `event_type` has the highest
total count across all windows?

In [ ]:
# Your turn (Exercise 1):

In [ ]:
# ✅ Solution 1
(spark.table("brewbox.events_hourly")
    .groupBy("event_type").agg(F.sum("count").alias("total"))
    .orderBy(F.desc("total")).show())

**Exercise 2 —** In words: why does a streaming **aggregation** need a
**watermark**? (Run the cell for the answer.)

In [ ]:
print("A watermark bounds how long Spark waits for late-arriving events before it "
      "finalizes (and emits) a time window, and lets it drop old state so memory "
      "stays bounded on an unbounded stream.")

**Exercise 3 —** What trigger would you use for a job that should process all new
files once per night and then stop? (Answer in the cell.)

In [ ]:
print("trigger(availableNow=True) - it processes everything available now, then "
      "stops, so you can schedule it nightly like a batch job while still getting "
      "incremental, checkpointed, exactly-once processing.")

## 7 · Recap & next

Structured Streaming reuses your batch logic to process data **continuously**:
`readStream`/`writeStream` + **checkpoints** + **triggers** + **watermarks**, with
`foreachBatch` for streaming upserts. Auto Loader makes file streaming trivial.

**Next → `12` Declarative pipelines (DLT / Lakeflow):** let Databricks manage the
streaming + batch plumbing declaratively, with built-in **data-quality
expectations**. 🚀